In [2]:
import win32com.client as com
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import numpy as np
from functools import lru_cache
from collections import defaultdict
from shapely.geometry import LineString
from shapely import wkt
import seaborn as sns

In [3]:
folder = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"

In [4]:
#Red base GDL (con links agregados por Johan y TALA)
red_base = os.path.join(folder, "Red Base GDL","RedBase 120826", "RedBase 300726 - conectores_final - finalfilt.ver")

import win32com.client
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = win32com.client.constants

## Open Sebastian shp Visum Links with attributes

In [7]:
# Read links from the final corrected shapefile
#links_red_final = gpd.read_file(os.path.join(folder, 'red_shapefiles', '10_red_final_corrected', 'red_final_corrected.gpkg'))
links_red_final = gpd.read_file(os.path.join(folder, 'red_shapefiles', '11_red_final_filtrada', 'filtered_final_network.gpkg'))

print(f"Links en la red final Sebastian: {len(links_red_final):,}")
#Probablemente son mas que los de Visum porque en Visum se borraron los links del macro que no servian 
# y yo solo le pase a Sebastian los los links nuevos de TALA y los trazos Johan
links_red_final["ageb_idx"] = (
    links_red_final["ageb_idx"]
    .fillna(-1)
    .astype(int)
)

# Read links from Visum
links_from_visum = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Links.GetMultiAttValues("No")],
    "from_node": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FromNodeNo")],
    "to_node": [i[1] for i in Visum.Net.Links.GetMultiAttValues("ToNodeNo")]
})
print(f"Links Visum Final: {len(links_from_visum):,}")

# Hacer merge on from node to node with Visum links
links_from_visum = links_from_visum.merge(
    links_red_final[["from_node", "to_node", "ageb_idx", "capacity", "num_lanes", "avg_vel", "limit_vel", "pop_dens", "job_dens", "dist_gdl", "boundary", "cluster", "subcluster", "topo_type", "topo_filt", "final_filter"]],
    on=["from_node", "to_node"],
    how="left"
)

Links en la red final Sebastian: 605,540
Links Visum Final: 605,326


In [8]:
links_red_final.columns

Index(['tsysset', 'highway', 'ageb_idx', 'from_node', 'to_node', 'u', 'v',
       'key', 'length', 'capacity', 'num_lanes', 'avg_vel', 'limit_vel',
       'pop_dens', 'job_dens', 'dist_gdl', 'boundary', 'cluster', 'subcluster',
       'topo_type', 'topo_filt', 'main_road', 'connector_end_link',
       'connector_intersected_link', 'connector_path', 'connector_backbone',
       'final_filter', 'geometry'],
      dtype='object')

In [10]:
# Para todos los 605,326 de la red de Visum (.ver final) hubo coincidencia
new_cols = [
    "ageb_idx", "capacity", "num_lanes", "avg_vel", "limit_vel",
    "pop_dens", "job_dens", "dist_gdl", "boundary",
    "cluster", "subcluster", "topo_type", "topo_filt", "final_filter"
]

links_from_visum[new_cols].isna().sum()

ageb_idx        0
capacity        0
num_lanes       0
avg_vel         0
limit_vel       0
pop_dens        0
job_dens        0
dist_gdl        0
boundary        0
cluster         0
subcluster      0
topo_type       0
topo_filt       0
final_filter    0
dtype: int64

In [11]:
links_from_visum.dtypes

No              float64
from_node       float64
to_node         float64
ageb_idx          int64
capacity          int64
num_lanes         int64
avg_vel         float64
limit_vel       float64
pop_dens        float64
job_dens        float64
dist_gdl        float64
boundary          int64
cluster           int64
subcluster        int64
topo_type         int64
topo_filt         int64
final_filter      int16
dtype: object

In [12]:
visum_links = Visum.Net.Links

links =  links_from_visum.copy()
links = links.reset_index(drop=True)
links.index = links.index + 1

"""attributes = {
    "CAPACIDAD_FINAL": ("capacity", int),
    "CARRILES_FINAL": ("num_lanes", int),
    "VELPROM_FINAL": ("avg_vel", float),
    "LIMVEL_FINAL": ("limit_vel", float),
    "POP_DENSITY": ("pop_dens", float),
    "JOB_DENSITY": ("job_dens", float),
    "DIST_TO_GDL": ("dist_gdl", float),
    "IS_BOUNDARY": ("boundary", int),
    "CLUSTER": ("cluster", int),
    "SUBCLUSTER": ("subcluster", int),
    "TOPO_TYPE": ("topo_type", int),
    "TOPO_FILT": ("topo_filt", int),
    "AGEB_IDX": ("ageb_idx", int),
}"""

attributes = {
    "FINAL_FILTER": ("final_filter", int)
}

for visum_att, (df_col, dtype) in attributes.items():
    values = list(
        zip(
            links.index,
            links[df_col].astype(dtype)
        )
    )

    visum_links.SetMultiAttValues(visum_att, values)

In [37]:
from_visum = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Links.GetMultiAttValues("No")],
    "from_node": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FromNodeNo")],
    "to_node": [i[1] for i in Visum.Net.Links.GetMultiAttValues("ToNodeNo")],
    "CapPrT": [i[1] for i in Visum.Net.Links.GetMultiAttValues("CAPPRT")],
    "V0PrT": [i[1] for i in Visum.Net.Links.GetMultiAttValues("V0PRT")],
    "NumLanes": [i[1] for i in Visum.Net.Links.GetMultiAttValues("NUMLANES")],
    "highway": [i[1] for i in Visum.Net.Links.GetMultiAttValues("HIGHWAY")],
})

print(from_visum['CapPrT'].value_counts().to_string())

CapPrT
2000.0     321012
0.0        138143
1000.0      58371
4000.0      36253
3000.0      26573
1500.0       7697
6000.0       7090
4500.0       5002
2500.0       2212
5000.0        971
2.0           519
8000.0        443
3500.0        238
5.0           138
10000.0       135
3750.0        106
12000.0        94
2250.0         75
9000.0         74
1750.0         64
1250.0         56
800.0          25
7500.0         16
7.0            11
3250.0          8


In [41]:
print(from_visum['V0PrT'].value_counts().sort_index().to_string())

V0PrT
0.00     138143
0.01        668
10.00        75
30.00    420301
40.00     39086
45.00        68
50.00      2032
55.00         1
60.00      1908
80.00      3044


In [56]:
pd.set_option('display.max_rows', None)

print(from_visum[from_visum['CapPrT']==0]['highway'].value_counts())

highway
                                                  96522
footway                                           21168
path                                               6160
cycleway                                           4195
track                                              3148
pedestrian                                         2756
steps                                               510
['footway', 'steps']                                493
['path', 'residential']                             428
['footway', 'residential']                          407
residential                                         305
['pedestrian', 'residential']                       207
['living_street', 'residential']                    196
['living_street', 'footway']                        176
corridor                                            150
['track', 'residential']                            137
service                                              93
unclassified                            